# 04 · Transformer extension: DistilBERT

**Owner:** Juan Diego Balanta (T7)

Fine-tunes `distilbert-base-uncased` for binary sentiment (0 = negative, 1 = positive) on the
balanced 500/500 GoEmotions subset, and evaluates it with the same five metrics as the other models
(accuracy, precision, recall, F1, Cohen's kappa).

Protocol (same rules as the Dense / RNN / LSTM notebook):
1. Hyperparameters are chosen on a **validation split taken from train** (15 %, stratified, seed 42).
2. The **test set is used only once**, for the final evaluation of the selected configuration.
3. The model reads the **raw** `text` column: DistilBERT has its own WordPiece tokenizer and was
   pre-trained on text with stopwords and punctuation, so we keep negations such as *not*.

> Run on **Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU). The whole grid takes a few
> minutes. On CPU it works but is slow.

In [ ]:
# --- Setup -----------------------------------------------------------------
# In Colab: upload/clone the repository and set REPO_DIR to its root.
# Example (private repo, use a GitHub personal access token):
#   !git clone https://<TOKEN>@github.com/AlejoRendon095/2026_2_Final_Proyect_discrete_structures.git repo
#   REPO_DIR = "repo"
import os, sys, json, time
from pathlib import Path

try:
    import transformers  # noqa: F401
except ImportError:
    %pip install -q transformers

REPO_DIR = Path(os.environ.get("REPO_DIR", "..")).resolve()
sys.path.insert(0, str(REPO_DIR))

DATA_DIR = REPO_DIR / "data" / "processed"
FIG_DIR = REPO_DIR / "outputs" / "figures"
MET_DIR = REPO_DIR / "outputs" / "metrics"
LOG_FILE = REPO_DIR / "logs" / "training_logs.txt"
for d in (FIG_DIR, MET_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
print("repo:", REPO_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.transformer_model import (TransformerConfig, train_transformer, predict_transformer,
                                   cls_attention, set_seed, get_device)

# Shared metric contract: evaluate(y_true, y_pred) -> dict with the 5 metrics (src/evaluate.py).
try:
    from src.evaluate import evaluate
except ImportError:  # fallback with the same contract, only while evaluate.py is not merged
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score
    def evaluate(y_true, y_pred):
        return {"accuracy": accuracy_score(y_true, y_pred),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "kappa": cohen_kappa_score(y_true, y_pred)}
    print("using local evaluate() fallback")

set_seed(SEED)
print("device:", get_device(), "| torch", torch.__version__)

## 1. Data

`train.csv` / `test.csv` follow the team data contract: columns `text`, `clean_text`, `label`,
800 / 200 rows, stratified 80/20 split with seed 42.

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
assert {"text", "label"} <= set(train_df.columns), "data contract: text, clean_text, label"

tr_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df["label"], random_state=SEED)
print(f"train {len(tr_df)} | val {len(val_df)} | test {len(test_df)}")
print("label balance (train):", tr_df["label"].value_counts().to_dict())

# Length in WordPiece tokens decides max_len (comments are short: <= 30 words).
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
lens = train_df["text"].map(lambda t: len(_tok(t)["input_ids"]))
print("WordPiece length: mean %.1f | p95 %d | max %d" % (lens.mean(), lens.quantile(.95), lens.max()))
MAX_LEN = int(min(128, max(32, np.ceil(lens.quantile(.99) / 8) * 8)))
print("MAX_LEN =", MAX_LEN)

## 2. Hyperparameter search (manual grid on the validation split)

Search space, following the usual BERT fine-tuning recommendations (Devlin et al., 2019):
learning rate ∈ {2e-5, 3e-5, 5e-5} × batch size ∈ {16, 32}. Each run trains for up to 4 epochs and keeps
the epoch with the best **validation macro-F1** (early stopping by checkpoint selection).

In [ ]:
GRID = [(lr, bs) for lr in (2e-5, 3e-5, 5e-5) for bs in (16, 32)]
EPOCHS = 4

rows = []
for lr, bs in GRID:
    cfg = TransformerConfig(learning_rate=lr, batch_size=bs, epochs=EPOCHS, max_len=MAX_LEN, seed=SEED)
    print(f"\n=== lr={lr}  batch={bs} ===")
    model, tok, hist = train_transformer(tr_df["text"], tr_df["label"], val_df["text"], val_df["label"], cfg)
    rows.append({"learning_rate": lr, "batch_size": bs, "best_epoch": hist["best_epoch"],
                 "val_macro_f1": hist["best_val_macro_f1"],
                 "val_accuracy": hist["val_accuracy"][hist["best_epoch"] - 1],
                 "train_time_sec": hist["train_time_sec"]})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

grid_df = pd.DataFrame(rows).sort_values(["val_macro_f1", "val_accuracy"], ascending=False)
grid_df.to_csv(MET_DIR / "transformer_grid_search.csv", index=False)
grid_df

## 3. Final model

Retrain the best configuration (same seed, so it reproduces the grid run) with the number of epochs it
selected, then evaluate **once** on the test set.

In [ ]:
best = grid_df.iloc[0]
final_cfg = TransformerConfig(learning_rate=float(best.learning_rate), batch_size=int(best.batch_size),
                              epochs=EPOCHS, max_len=MAX_LEN, seed=SEED)
model, tok, history = train_transformer(tr_df["text"], tr_df["label"], val_df["text"], val_df["label"], final_cfg)

train_pred, _ = predict_transformer(model, tok, train_df["text"], max_len=MAX_LEN)
test_pred, test_prob = predict_transformer(model, tok, test_df["text"], max_len=MAX_LEN)

results = {
    "model": "DistilBERT (distilbert-base-uncased, fine-tuned)",
    "train": {k: float(v) for k, v in evaluate(train_df["label"], train_pred).items()},
    "test": {k: float(v) for k, v in evaluate(test_df["label"], test_pred).items()},
    "best_epoch": history["best_epoch"],
    "train_time_sec": history["train_time_sec"],
    "parameters": history["parameters"],
    "device": history["device"],
    "config": history["config"],
    "history": {k: history[k] for k in ("train_loss", "val_loss", "val_accuracy", "val_macro_f1")},
}
with open(MET_DIR / "transformer_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

pd.DataFrame({"train": results["train"], "test": results["test"]}).round(4)

In [ ]:
# Learning curves of the final run
ep = range(1, len(history["train_loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ep, history["train_loss"], "o-", label="train")
ax[0].plot(ep, history["val_loss"], "o-", label="validation")
ax[0].set(title="DistilBERT loss", xlabel="epoch", ylabel="cross-entropy"); ax[0].legend()
ax[1].plot(ep, history["val_accuracy"], "o-", label="val accuracy")
ax[1].plot(ep, history["val_macro_f1"], "o-", label="val macro-F1")
ax[1].axvline(history["best_epoch"], ls="--", c="gray", lw=1)
ax[1].set(title="Validation metrics", xlabel="epoch"); ax[1].legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "transformer_learning_curves.png", dpi=200)
plt.show()

cm = confusion_matrix(test_df["label"], test_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"])
disp.plot(cmap="Blues", colorbar=False); plt.title("DistilBERT - test confusion matrix")
plt.tight_layout(); plt.savefig(FIG_DIR / "transformer_confusion_matrix.png", dpi=200); plt.show()

## 4. What does attention look at?

Attention from the `[CLS]` token in the last layer (mean over the 12 heads). `[CLS]` is the vector the
classifier reads, so this is a rough, **non-causal** hint of which tokens feed the decision
(attention weights are not explanations by themselves, Jain & Wallace, 2019).

In [ ]:
def plot_cls_attention(text, ax):
    tokens, att = cls_attention(model, tok, text, max_len=MAX_LEN)
    tokens, att = tokens[1:-1], att[1:-1]           # drop [CLS] and [SEP]
    ax.bar(range(len(tokens)), att)
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=60, ha="right", fontsize=8)
    ax.set_ylabel("[CLS] attention")

correct = test_df.assign(pred=test_pred, prob=test_prob)
examples = [correct[(correct.label == 1) & (correct.pred == 1)].sort_values("prob").iloc[-1],
            correct[(correct.label == 0) & (correct.pred == 0)].sort_values("prob").iloc[0]]
fig, axes = plt.subplots(len(examples), 1, figsize=(10, 3 * len(examples)))
for ax, row in zip(axes, examples):
    plot_cls_attention(row.text, ax)
    ax.set_title(f"label={'positive' if row.label else 'negative'}  p(pos)={row.prob:.2f}", fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR / "transformer_attention_examples.png", dpi=200)
plt.show()

## 5. Error analysis

In [ ]:
errors = correct[correct.label != correct.pred].copy()
errors["confidence"] = np.where(errors.pred == 1, errors.prob, 1 - errors.prob)
print(f"{len(errors)} errors out of {len(correct)} test comments")
pd.set_option("display.max_colwidth", 120)
errors.sort_values("confidence", ascending=False)[["text", "label", "pred", "confidence"]].head(10)

In [ ]:
# Append a summary line to the shared training log
with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(f"[{time.strftime('%Y-%m-%d %H:%M')}] DistilBERT | {json.dumps(results['config'])} | "
            f"best_epoch={results['best_epoch']} | time={results['train_time_sec']}s | "
            f"test={json.dumps({k: round(v, 4) for k, v in results['test'].items()})}\n")
print("logged")

## 6. Discussion (to complete with the final numbers)

- **Performance vs recurrent models:** compare test F1 / kappa with the RNN and LSTM
  (`outputs/metrics/`), and the train–test gap as a sign of overfitting.
- **Attention vs recurrence:** the RNN/LSTM compress the comment left to right into a fixed-size state
  (a bounded, sequential memory, like reading a tape once). Self-attention lets every token read every other
  token in one step, so there is no vanishing-gradient path through time and long-range cues
  (e.g. *"not ... good"*) are directly connected.
- **Cost:** ~66 M parameters vs a few hundred thousand for the recurrent models; transfer learning is what
  makes it work with only 680 training comments.